In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os, sys

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -----------------------------
# Add shared utilities path
# -----------------------------
NOTEBOOK_DIR = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
BUNDLE_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_shared_utilities")

from superstore_platform_config import get_catalog, get_gold_schema, get_mart_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
gold_schema = get_gold_schema()    # e.g., dev_gold
mart_schema = get_mart_schema() 

from superstore_logger import get_superstore_logger, log_event
from superstore_backfill_utils import is_dry_run
# -----------------------------
# log_event Setup 
# -----------------------------
# Initialize log_event to capture events 
logger_mart_sales_daily = get_superstore_logger("superstore_mart_sales_daily")


# -----------------------------
# Dry run
# -----------------------------
# The serving layer rebuilds from Gold and has no run-mode logic of its own —
# a replay or backfill simply produces different Gold for it to read. But it
# does write, so it must not write during a dry run. is_dry_run reads only the
# dry_run parameter and never raises, so a serving task cannot fail on the
# full_refresh gate that is enforced upstream where it belongs.
if is_dry_run(dbutils):
    log_event(
        logger_mart_sales_daily,
        "INFO",
        "Dry run: skipping mart_sales_daily write",
        dry_run=True,
        layer="Mart"
    )
    dbutils.notebook.exit("DRY_RUN_COMPLETED layer=mart_sales_daily (no data written)")

# -------------------------------
# Function to check if the table exists
# -------------------------------
def check_table_exists(catalog, schema, table_name):
    try:
        full_table_name = f"{catalog}.{schema}.{table_name}"
        return spark.catalog.tableExists(full_table_name)
    except Exception as e:
        log_event(
            logger_mart_sales_daily,
            "ERROR",
            f"Error checking table existence: {e}",
            layer="Feature Engineering"
        )
        return False

# -------------------------------
# Required table to create or update features Table (Environment-aware)
# -------------------------------
# Check if the required tables exist before creating the feature table
required_tables = [
    f"{catalog}.{gold_schema}.dim_customers",
    f"{catalog}.{gold_schema}.facts_orders",
    f"{catalog}.{gold_schema}.facts_sales"
]

missing_tables = []

# Check each table
for table in required_tables:
    if not check_table_exists(catalog, gold_schema, table.split('.')[-1]):
        missing_tables.append(table)

if missing_tables:
    log_event(logger_mart_sales_daily, "ERROR", f"Missing tables: {', '.join(missing_tables)}. Cannot create mart sales daily table.", layer="Mart")
else:
  # -------------------------------
  # Referential completeness
  # -------------------------------
  # This mart is built from Gold facts and dimensions joined on the CURRENT
  # dimension version. A fact whose dimension row is absent contributes nothing
  # to the mart, and nothing reports it: the row is not quarantined, not
  # audited, and not counted anywhere. It is simply not in the output.
  #
  # The reconciliation invariant does not cover this. It proves
  # bronze == silver + quarantine + audit, which is a SILVER guarantee that
  # every row is accounted for - not that it is usable downstream. A customer
  # quarantined for one bad attribute never reaches the dimension, so their
  # orders are dropped here while every upstream check still passes.
  #
  # Counting it does not fix it. It makes a silent loss a monitored one, which
  # is the same reason quarantine records the rule a row violated.

  _orphans_customers = spark.sql(f"""
      SELECT COUNT(*) AS n
      FROM {catalog}.{gold_schema}.facts_orders f
      LEFT ANTI JOIN {catalog}.{gold_schema}.dim_customers d
        ON f.customer_id = d.customer_id AND d.is_current = true
  """).first()["n"]

  log_event(
      logger_mart_sales_daily,
      "WARN" if _orphans_customers else "INFO",
      f"facts_orders rows with no current dim_customers version: {_orphans_customers}",
      orphaned_facts=_orphans_customers,
      fact_table="facts_orders",
      dimension_table="dim_customers",
      layer="Mart"
  )

  _orphans_products = spark.sql(f"""
      SELECT COUNT(*) AS n
      FROM {catalog}.{gold_schema}.facts_sales f
      LEFT ANTI JOIN {catalog}.{gold_schema}.dim_products d
        ON f.product_id = d.product_id AND d.is_current = true
  """).first()["n"]

  log_event(
      logger_mart_sales_daily,
      "WARN" if _orphans_products else "INFO",
      f"facts_sales rows with no current dim_products version: {_orphans_products}",
      orphaned_facts=_orphans_products,
      fact_table="facts_sales",
      dimension_table="dim_products",
      layer="Mart"
  )

  # -------------------------------
  # Create mart Table (Environment-aware)
  # -------------------------------
  spark.sql(f"""
  CREATE OR REPLACE TABLE {catalog}.{mart_schema}.mart_sales_daily AS
  SELECT
    o.order_date,
    o.customer_id,
    s.product_id,
    c.region,
    c.segment,
    p.category,
    p.sub_category,
    o.ship_mode,

    SUM(s.sales) AS total_sales,
    SUM(s.profit) AS total_profit,
    SUM(s.quantity) AS total_quantity,
    AVG(s.discount) AS avg_discount,
    COUNT(DISTINCT o.order_id) AS order_count,

    CASE 
    WHEN SUM(s.sales) = 0 THEN NULL 
    ELSE ROUND(SUM(s.profit) / SUM(s.sales), 4) 
    END AS profit_margin

  FROM {catalog}.{gold_schema}.facts_orders o
  JOIN {catalog}.{gold_schema}.facts_sales s 
  ON o.order_id = s.order_id

  -- FIXED: use o.customer_id
  JOIN {catalog}.{gold_schema}.dim_customers c 
      ON o.customer_id = c.customer_id 
      AND c.is_current = true

  JOIN {catalog}.{gold_schema}.dim_products p 
    ON s.product_id = p.product_id 
    AND p.is_current = true

  GROUP BY
    o.order_date,
    o.customer_id,
    s.product_id,
    c.region,
    c.segment,
    p.category,
    p.sub_category,
    o.ship_mode
  ;
  """)

log_event(logger_mart_sales_daily, "INFO", "Mart mart sales daily table created successfully.", layer="Mart")
